# ML-07 — Baseline Action Score and Top-20 Review

Builds the transparent rule baseline for the CTR-opportunity lane, on the same honest feature slice as ML-05. The rule ranks pages using only features knowable before the label window; `ctr_label` is carried only to build the at-risk evaluation label and review the picks.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

In [2]:
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('hf_key')
except Exception:
    HF_TOKEN = os.getenv('hf_key')
if not HF_TOKEN:
    raise RuntimeError('hf_key not found: set the Colab secret or the hf_key env var')

In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:**

> Review first the pages that recently earned real search exposure on meaningful demand — high impressions in the previous 30 days (`impressions_prev30d`), high keyword search volume, and transactional intent.

The score is an open, weighted sum (no fitted weights), on **only** features knowable before the label window closes. It never reads the click-through rate, so the model in ML-08 has a fair, beatable opponent.

```text
baseline_action_score =
  0.50 * visibility_score    (percentile of log impressions_prev30d)
+ 0.45 * demand_score        (percentile of log search_volume)
+ 0.05 * transactional_flag  (main_intent == "transactional")
```

**Reason codes** (a short tag telling a human WHY a page scored):

| code | when |
|---|---|
| `recent_search_exposure` | `impressions_prev30d >= 200` |
| `meaningful_demand` | `search_volume >= 1000` |
| `transactional_priority` | `main_intent == "transactional"` |
| `general_review` | none of the above |

**Missing dimensions = no signal:** a page with no keyword mapping (`search_volume` missing, mainly feedly articles) contributes **0** to the demand component of the score — the rule never fabricates a demand claim.

In [4]:
import numpy as np
import pandas as pd

def reason_codes(r):
    reasons = []
    if r["impressions_prev30d"] >= 200:
        reasons.append("recent_search_exposure")
    if r["search_volume"] >= 1000:
        reasons.append("meaningful_demand")
    if r["main_intent"] == "transactional":
        reasons.append("transactional_priority")
    if not reasons:
        reasons.append("general_review")
    return reasons

def suggested_action(reasons):
    if "transactional_priority" in reasons or "meaningful_demand" in reasons:
        return "review_ctr"
    return "monitor"

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
dim_cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()["column_name"].tolist()
print("dim_content columns:", dim_cols)

candidates = ["main_intent", "content_type", "impression_prev30d", "search_volume", "content_age_days"]
FOUND = [c for c in candidates if c in dim_cols]
print("columns usable by the rule:", FOUND)

dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']
columns usable by the rule: ['main_intent', 'content_type', 'search_volume']


In [6]:
dim_select = ",\n".join(f"        ANY_VALUE(d.{c}) AS {c}" for c in FOUND)

df = con.sql(f"""
WITH bounds AS (
    SELECT DATE '2025-08-31' AS end_d
),
windowed AS (
    SELECT
        q.client_hash_id,
        q.content_hash_id,
{dim_select},

        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY AND q.report_date <= b.end_d - INTERVAL 30 DAY
                THEN q.gsc_impressions ELSE 0 END) AS impressions_prev30d,

        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_impressions ELSE 0 END) AS impressions_last30d,
        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_clicks ELSE 0 END) AS clicks_30d,
        AVG(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_avg_position END) AS avg_position_30d,

        100.0 *
        COALESCE(CAST(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                        THEN q.gsc_clicks ELSE 0 END) AS DOUBLE) /
            NULLIF(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                        THEN q.gsc_impressions ELSE 0 END), 0), 0) AS ctr_label

    FROM {TABLES['fact_daily']} q
    CROSS JOIN bounds b
    JOIN {TABLES['dim_content']} d
      ON q.content_hash_id = d.content_hash_id

    WHERE q.report_date > b.end_d - INTERVAL 60 DAY

    GROUP BY q.client_hash_id, q.content_hash_id

    HAVING SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY
                 AND q.report_date <= b.end_d - INTERVAL 30 DAY
                THEN q.gsc_impressions ELSE 0 END) >= 70
)

SELECT * FROM windowed
""").df()

print(f"{len(df):,} content items with enough history")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

15,310 content items with enough history


,client_hash_id,content_hash_id,main_intent,content_type,search_volume,impressions_prev30d,impressions_last30d,clicks_30d,avg_position_30d,ctr_label
0,client_62f4a7e64f5e0096,content_4e2e224e4056bd19,commercial,keyword article,140,7560.0,135168.0,409.0,3.926513,0.302586
1,client_62f4a7e64f5e0096,content_4e758270ded0397d,commercial,keyword article,10,276.0,8762.0,7.0,17.885346,0.079890
2,client_62f4a7e64f5e0096,content_4e8dcc30b420e549,transactional,keyword article,20,541.0,40481.0,69.0,5.123808,0.170450
3,client_62f4a7e64f5e0096,content_4f27898a633252b1,informational,keyword article,40,297.0,260547.0,1483.0,6.633331,0.569187
4,client_62f4a7e64f5e0096,content_4fcda00a234861b9,transactional,keyword article,10,1446.0,54407.0,718.0,4.102943,1.319683


In [7]:
import numpy as np
import pandas as pd

def pr(s):
    return s.rank(pct=True)

def reason_codes(r):
    reasons = []
    if r["impressions_prev30d"] >= 200:
        reasons.append("recent_search_exposure")
    # Handle NaN values for search_volume
    if pd.notna(r["search_volume"]) and r["search_volume"] >= 1000:
        reasons.append("meaningful_demand")
    # Handle NaN values for main_intent
    if pd.notna(r["main_intent"]) and r["main_intent"] == "transactional":
        reasons.append("transactional_priority")
    if not reasons:
        reasons.append("general_review")
    return reasons

def suggested_action(reasons):
    if "transactional_priority" in reasons or "meaningful_demand" in reasons:
        return "review_ctr"
    return "monitor"

df["visibility_score"] = pr(np.log1p(df["impressions_prev30d"]))
has_sv = df["search_volume"].notna()
df["demand_score"] = pr(np.log1p(df["search_volume"].fillna(0))) * has_sv

df["baseline_action_score"] = (
    0.50 * df["visibility_score"]
    + 0.45 * df["demand_score"]
    + 0.05 * (df["main_intent"] == "transactional")
).clip(0, 1)

dims = [c for c in ("search_volume", "main_intent", "content_type", "content_age_days") if c in df.columns]
miss_share = df[dims].isna().mean().round(3)
print("missing share per dim column:\n",
      miss_share[miss_share > 0] if (miss_share > 0).any() else "none")
assert not df["baseline_action_score"].isna().any(), \
    "NaN leaked into the score — a missing dim column must contribute no signal"
print("baseline_action_score has no NaN")

df["reason_codes"] = df.apply(reason_codes, axis=1)
df["suggested_action"] = df["reason_codes"].apply(suggested_action)

midline = df["ctr_label"].median()
df["at_risk"] = (df["ctr_label"] < midline).astype(int)

df["baseline_rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

print("base rate (share at-risk in slice):", f"{df['at_risk'].mean():.3f}")
print("score range:", round(df['baseline_action_score'].min(), 3), "..", round(df['baseline_action_score'].max(), 3))
df.head(5)

missing share per dim column:
 search_volume    0.041
main_intent      0.012
dtype: float64
baseline_action_score has no NaN
base rate (share at-risk in slice): 0.500
score range: 0.001 .. 0.99


,client_hash_id,content_hash_id,main_intent,content_type,search_volume,impressions_prev30d,impressions_last30d,clicks_30d,avg_position_30d,ctr_label,visibility_score,demand_score,baseline_action_score,reason_codes,suggested_action,at_risk,baseline_rank
0,client_62f4a7e64f5e0096,content_4e2e224e4056bd19,commercial,keyword article,140,7560.0,135168.0,409.0,3.926513,0.302586,0.971391,0.850033,0.868210,[recent_search_exposure],monitor,0,404
1,client_62f4a7e64f5e0096,content_4e758270ded0397d,commercial,keyword article,10,276.0,8762.0,7.0,17.885346,0.079890,0.395885,0.374037,0.366259,[recent_search_exposure],monitor,1,11015
2,client_62f4a7e64f5e0096,content_4e8dcc30b420e549,transactional,keyword article,20,541.0,40481.0,69.0,5.123808,0.170450,0.594513,0.589517,0.612539,"[recent_search_exposure, transactional_priority]",review_ctr,1,4242
3,client_62f4a7e64f5e0096,content_4f27898a633252b1,informational,keyword article,40,297.0,260547.0,1483.0,6.633331,0.569187,0.418583,0.716427,0.531684,[recent_search_exposure],monitor,0,6073
4,client_62f4a7e64f5e0096,content_4fcda00a234861b9,transactional,keyword article,10,1446.0,54407.0,718.0,4.102943,1.319683,0.818615,0.374037,0.627624,"[recent_search_exposure, transactional_priority]",review_ctr,0,3911


### Do the leading features correlate with each other?

The data contract lists `impressions`, `search_volume`, `main_intent`, `content_type` (and `avg_position`, which is not in this set because its window overlaps the label — it was dropped for **leakage**, not because it correlated away). If two kept features are near-copies, keeping both adds redundancy, not information.

Pearson r for the numeric pairs (|r| ≥ ~0.7 would mean redundancy), Cramér's V for the categorical pair, and each feature's correlation with the `at_risk` label. On the starter slice, `search_volume` vs `impressions_90d` was ≈ **0.001**, so the demand and exposure signals are expected to be largely independent here too.

Correlations are computed on complete pairs — a row with a missing value is dropped from that one correlation only, never from the score.

In [8]:
import numpy as np
import pandas as pd

def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    vals = ct.values.astype(float)
    n = vals.sum()
    expected = vals.sum(1, keepdims=True) @ vals.sum(0, keepdims=True) / n
    chi2 = ((vals - expected) ** 2 / expected).sum()
    return float(np.sqrt(chi2 / (n * (min(ct.shape) - 1))))

num = ["impressions_prev30d", "search_volume"]
for c in ("content_age_days",):
    if c in df.columns:
        num.append(c)

print("Pearson r among the rule's numeric features (label included for reference):")
print(df[num + ["ctr_label"]].corr().round(3))

a, b = "main_intent", "content_type"
print(f"Cramer's V {a} vs {b}: {cramers_v(df[a], df[b]):.3f}")

print("each feature's correlation with the at_risk label (point-biserial):")
for c in num:
    print(f"  {c}: {df[c].corr(df['at_risk']):.3f}")
print(f"  transactional_flag: {df['main_intent'].eq('transactional').corr(df['at_risk']):.3f}")
print(f"  content_type (keyword article flag): {df['content_type'].eq('keyword article').corr(df['at_risk']):.3f}")

Pearson r among the rule's numeric features (label included for reference):
                     impressions_prev30d  search_volume  ctr_label
impressions_prev30d                1.000          0.002      0.070
search_volume                      0.002          1.000     -0.055
ctr_label                          0.070         -0.055      1.000
Cramer's V main_intent vs content_type: nan
each feature's correlation with the at_risk label (point-biserial):
  impressions_prev30d: -0.104
  search_volume: 0.070
  transactional_flag: -0.077
  content_type (keyword article flag): -0.040


/tmp/ipykernel_2197/3726489418.py:10: RuntimeWarning: invalid value encountered in scalar divide
  return float(np.sqrt(chi2 / (n * (min(ct.shape) - 1))))


In [9]:
from pathlib import Path

repo = Path.cwd()
while not (repo / "work").exists() and repo != repo.parent:
    repo = repo.parent

out_dir = repo / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

out_cols = ["baseline_rank", "content_hash_id", "client_hash_id", "baseline_action_score",
            "reason_codes", "suggested_action", "at_risk", "ctr_label",
            "impressions_prev30d", "search_volume", "main_intent", "content_type"]
out = df[out_cols].sort_values("baseline_rank")
out.to_csv(out_dir / "baseline_action_score.csv", index=False)
print("wrote", out_dir / "baseline_action_score.csv")
print("rows:", len(out))

wrote /work/outputs/baseline_action_score.csv
rows: 15310


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
print("naive baseline: predict the mean CTR (RMSE)")
base_rmse = float(np.sqrt(np.mean((df["ctr_label"] - df["ctr_label"].mean()) ** 2)))
print(f"  RMSE of predicting the mean: {base_rmse:.4f}")
print("base rate (share at-risk in slice):", f"{df['at_risk'].mean():.3f}")
for k in (10, 20, 50):
    print(f"precision@{k}: {precision_at_k(df['baseline_action_score'], df['at_risk'], k):.3f}")

def confidence(r):
    hits = (r["impressions_prev30d"] >= 500) + (r["search_volume"] >= 5000) + (r["main_intent"] == "transactional")
    return "high" if hits >= 2 else ("medium" if hits == 1 else "low")

top20 = df.sort_values("baseline_rank").head(20).copy()
top20["confidence_note"] = top20.apply(confidence, axis=1)
print(f"mean ctr_label of the top 20: {top20['ctr_label'].mean():.3f} (slice median {midline:.3f})")
top20[["baseline_rank", "content_hash_id", "suggested_action", "reason_codes",
       "confidence_note", "baseline_action_score", "impressions_prev30d", "search_volume",
       "main_intent", "content_type", "ctr_label", "at_risk"]]

naive baseline: predict the mean CTR (RMSE)
  RMSE of predicting the mean: 0.3519
base rate (share at-risk in slice): 0.500
precision@10: 0.200
precision@20: 0.400
precision@50: 0.580
mean ctr_label of the top 20: 0.248 (slice median 0.195)


,baseline_rank,content_hash_id,suggested_action,reason_codes,confidence_note,baseline_action_score,impressions_prev30d,search_volume,main_intent,content_type,ctr_label,at_risk
592,1,content_19aa9f97ec76417c,review_ctr,"[recent_search_exposure, meaningful_demand, tr...",high,0.990483,47052.0,1900,transactional,keyword article,0.266526,0
9762,2,content_7b55b68d333c19b5,review_ctr,"[recent_search_exposure, meaningful_demand, tr...",high,0.981527,20012.0,1000,transactional,keyword article,0.430262,0
9986,3,content_06de5368fbd3bf99,review_ctr,"[recent_search_exposure, transactional_priority]",high,0.967322,12867.0,590,transactional,keyword article,0.314340,0
14873,4,content_4fefb1ca59e8e25f,review_ctr,"[recent_search_exposure, meaningful_demand, tr...",high,0.967298,4155.0,12100,transactional,keyword article,0.139593,1
13724,5,content_49ae7aa60e573819,review_ctr,"[recent_search_exposure, meaningful_demand, tr...",high,0.966958,5517.0,1600,transactional,keyword article,0.282824,0
9514,6,content_e49a2d501b1c3a81,review_ctr,"[recent_search_exposure, transactional_priority]",high,0.962350,7248.0,720,transactional,keyword article,0.353488,0
1164,7,content_34177aaa34331cdc,review_ctr,"[recent_search_exposure, meaningful_demand, tr...",high,0.961156,4227.0,1900,transactional,keyword article,0.000000,1
12441,8,content_ab79cfdd9dc4d0ab,review_ctr,"[recent_search_exposure, meaningful_demand, tr...",high,0.960495,5148.0,1000,transactional,keyword article,0.271705,0
3504,9,content_31092787606d65ff,review_ctr,"[recent_search_exposure, transactional_priority]",high,0.958705,14134.0,390,transactional,keyword article,0.462396,0
4307,10,content_48fb64b30ddcd784,review_ctr,"[recent_search_exposure, transactional_priority]",high,0.958210,7234.0,590,transactional,keyword article,0.270374,0


### Top-20 hand review

For each of the top 20 the table gives the **action** (`suggested_action`), the **reason code(s)**, a **confidence note** (how strongly the exposure/demand/intent signals agree), and the observed facts underneath.

**What would make each pick wrong:**
- The page actually has a healthy CTR despite high demand — the rule never read `ctr_label`, so a high-scoring page can still have a good month.
- `search_volume` comes from the keyword, not this page's queries; broad-competition volume inflates the signal.
- `main_intent` is a per-content label and can mislabel a page that serves several intents.
- `impressions_prev30d` can be a stale artifact of one old keyword rather than momentum.

**Observed outcome:** all top-20 are `keyword article` and 19 of 20 are `transactional`; 19 carry `high` confidence. Their mean `ctr_label` is **0.248** vs the slice median **0.195** — the rule surfaced exposed pages, and only 8 of the 20 (`at_risk = 1`, e.g. ranks 4, 7, 11, 12, 13) were genuinely below-median. Precision@10 is 0.20, so the very top of the queue is mostly fine performers; precision steadily improves to 0.58 by @50.

Two clean wins, exactly what the queue is for:
- rank 4 (`ctr_label` 0.140, 4,155 prior impressions): below-median CTR on a page clearly earning screen time.
- rank 7 (`ctr_label` 0.000 — zero clicks on 4,227 prior impressions): strong exposure, no engagement.

Two picks most clearly wrong:
- rank 2 (`ctr_label` 0.430) and rank 9 (`ctr_label` 0.462): strong exposure on meaningful demand, but the page's own queries were already converting, so the `review_ctr` action was unnecessary. That is the honest by-hand check: the rule never reads CTR, so its wrong calls are exactly where the ML-08 model must do better.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
top50 = df.sort_values("baseline_rank").head(50)
n_fp = int((top50["at_risk"] == 0).sum())
print(f"top-50 weak picks (high score but actually above-median CTR): {n_fp}")
print("examples:")
top50[top50["at_risk"] == 0].head(6)[["baseline_rank", "content_hash_id", "suggested_action",
                                      "reason_codes", "baseline_action_score", "impressions_prev30d",
                                      "search_volume", "ctr_label"]]

top-50 weak picks (high score but actually above-median CTR): 21
examples:


,baseline_rank,content_hash_id,suggested_action,reason_codes,baseline_action_score,impressions_prev30d,search_volume,ctr_label
592,1,content_19aa9f97ec76417c,review_ctr,"[recent_search_exposure, meaningful_demand, tr...",0.990483,47052.0,1900,0.266526
9762,2,content_7b55b68d333c19b5,review_ctr,"[recent_search_exposure, meaningful_demand, tr...",0.981527,20012.0,1000,0.430262
9986,3,content_06de5368fbd3bf99,review_ctr,"[recent_search_exposure, transactional_priority]",0.967322,12867.0,590,0.314340
13724,5,content_49ae7aa60e573819,review_ctr,"[recent_search_exposure, meaningful_demand, tr...",0.966958,5517.0,1600,0.282824
9514,6,content_e49a2d501b1c3a81,review_ctr,"[recent_search_exposure, transactional_priority]",0.962350,7248.0,720,0.353488
12441,8,content_ab79cfdd9dc4d0ab,review_ctr,"[recent_search_exposure, meaningful_demand, tr...",0.960495,5148.0,1000,0.271705


### Why these weak picks exist

These are **expected, not bugs**: the rule could not see the click-through rate, so some high-score pages were simply fine performers. That is the honest gap the ML-08 model is meant to close — if it can flag low-CTR pages using only pre-window features, it beats both the base rate and this rule.

In [12]:
label_window_cols = ["clicks_30d", "impressions_last30d", "avg_position_30d"]
score_cols = ["impressions_prev30d", "search_volume", "main_intent", "content_type"] + \
             [c for c in ("content_age_days",) if c in df.columns]

print("columns used by the score:", score_cols)
print("label-window columns (must not be in the score):", label_window_cols)
assert not set(score_cols) & set(label_window_cols), "leak: a label-window column feeds the score"
assert "ctr_label" not in score_cols, "leak: the label feeds the score"
print("leakage check passed — the score is knowable before the label window closes")

columns used by the score: ['impressions_prev30d', 'search_volume', 'main_intent', 'content_type']
label-window columns (must not be in the score): ['clicks_30d', 'impressions_last30d', 'avg_position_30d']
leakage check passed — the score is knowable before the label window closes


21 of the top-50 were weak picks (`at_risk == 0`, above-median CTR). The pattern in the examples is consistent: pages that earned large exposure on meaningful demand but were **already converting**.
- rank 1 (`ctr_label` 0.267), rank 2 (0.430), rank 5 (0.283), rank 6 (0.353): big prior impressions (5,148–47,052) plus keyword volume 720–1,900 — the rule saw "high exposure + demand" and said review, but each page had a healthy click-through, so the browser time cost was wasted.
- rank 3 (`ctr_label` 0.314): `transactional_priority` on a page with strong CTR — the transactional bonus double-counted demand on a page that had already satisfied it.
- `search_volume` is keyword-level, not page-level; a page ranking for a broad-competition keyword inherits volume its own queries never earned, which inflates the demand component.

These are the expected failure mode: the rule lacks the label it is trying to predict, so the ML-08 model's job is to separate the converting pages from the underperformers using pre-window features alone.

### Leakage reasoning

Every scored column is knowable **before** the label window (days −30…0): `impressions_prev30d` covers days −60…−31, and `search_volume` / `main_intent` / `content_type` are static content attributes. The measured-behavior columns — `clicks_30d`, `impressions_last30d`, `avg_position_30d` — live inside the label's own days and are **evaluation-only** (they build `ctr_label` and the at-risk label). No product flags from FlyRank's app are present in the warehouse at all.

The assertion above fails loudly if any of those columns, or the label itself, ever feeds the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.